# AI Essay Detector - Step 2: Preprocessing & Feature Extraction

This notebook performs Stratified 5-Fold Splitting and extracts hybrid TF-IDF features (combining word-level and character-level features) without data leakage.


### Kaggle Dataset Presence Check & Downloader
This block checks if the dataset `train_v4_drcat_01.csv` is present locally. If not, it sets up Kaggle, downloads the dataset zip, extracts it, and performs cleanup.

In [ ]:
import os

DATASET_FILE_NAME = "train_v4_drcat_01.csv"
KAGGLE_ZIP_FILE_NAME = "daigt-v4-train-dataset.zip"

# Check if the dataset file already exists locally
if os.path.exists(DATASET_FILE_NAME):
    print(f"'{DATASET_FILE_NAME}' found locally. Skipping Kaggle download.")
else:
    print(f"'{DATASET_FILE_NAME}' not found locally. Attempting to download from Kaggle...")

    # 1. Install the Kaggle library
    !pip install kaggle --quiet

    # Ensure the .kaggle directory exists (redundant if previous cell ran, but safe)
    !mkdir -p ~/.kaggle

    # The kaggle.json file should have been created by the previous cell
    # To prevent 'kaggle.json' from being publicly readable
    !chmod 600 ~/.kaggle/kaggle.json

    print("Kaggle API setup should be complete (kaggle.json created by previous cell).")

    # 3. Download the dataset
    dataset_name = "thedrcat/daigt-v4-train-dataset"
    !kaggle datasets download -d {dataset_name}

    # 4. Unzip the downloaded file
    !unzip -o {KAGGLE_ZIP_FILE_NAME} -d .

    # 5. Clean up the zip file (optional)
    !rm {KAGGLE_ZIP_FILE_NAME}

    if os.path.exists(DATASET_FILE_NAME):
        print(f"\nDataset '{dataset_name}' downloaded and unzipped successfully.")
        print(f"You should now see '{DATASET_FILE_NAME}' in your current directory.")
    else:
        raise FileNotFoundError(f"Failed to download and extract '{DATASET_FILE_NAME}' from Kaggle.")


### Working Directory & Path Initialization
Since this notebook runs inside the `notebooks/` directory, we shift the kernel's active path to the project root (`..`) so that relative data paths map correctly. We also add `src/` to the python import path so that custom helper modules like `evaluate` can be imported.

In [ ]:
import os
import sys

# If running from notebooks/ directory, switch to the project root
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

# Ensure the src/ directory is in the import path so evaluate can be found
sys.path.append(os.path.abspath('src'))

print(f"Current Working Directory set to: {os.getcwd()}")
print("Import path appended. You can now import evaluation utilities successfully.")


### 1. Import Feature Extraction & Cross-Validation Packages
We import cross-validation modules (`StratifiedKFold`), vectorizers (`TfidfVectorizer`), and sparse matrix helper library (`scipy.sparse.hstack`).


In [ ]:
import os
import joblib
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack, save_npz


### 2. Preprocessing Function
This performs feature engineering and split generation:
- **Stratified K-Fold**: Splits the cleaned data into 5 folds while maintaining class balance.
- **Zero-Data Leakage Preprocessing**: For each fold, we fit vectorizers *only* on the training fold, and then transform the test fold.
- **Hybrid Text Representation**: We fit both word-level TF-IDF (1-2 words) and character-level TF-IDF (3-5 characters) vectorizers and stack them horizontally into a 50,000 column space.
- **Export**: Exports the sparse matrices and target labels to the `processed_data/` folder, and vectorizers to the `models/` folder.


In [ ]:
def run_preprocessing(input_csv, output_dir, models_dir):
    print("=== Loading Cleaned Dataset ===")
    if not os.path.exists(input_csv):
        raise FileNotFoundError(f"Cleaned dataset not found at {input_csv}. Please run clean_dataset.py first.")
        
    df = pd.read_csv(input_csv)
    print(f"Loaded {df.shape[0]} rows.")
    
    # Fill any NaNs just in case
    df['text'] = df['text'].fillna('')
    
    print("\n=== Performing 5-Fold Stratified Split ===")
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    for fold, (train_idx, test_idx) in enumerate(skf.split(df, df['label'])):
        print(f"\n--- Processing Fold {fold} ---")
        train_df = df.iloc[train_idx].copy()
        test_df = df.iloc[test_idx].copy()
        
        print(f"Training set size: {train_df.shape[0]} rows")
        print(f"Testing set size: {test_df.shape[0]} rows")
        
        # Track label distributions
        for dataset_name, subset in [("Train", train_df), ("Test", test_df)]:
            counts = subset['label'].value_counts()
            print(f"  {dataset_name} set - AI (1): {counts.get(1, 0)} ({counts.get(1, 0)/len(subset)*100:.2f}%), "
                  f"Human (0): {counts.get(0, 0)} ({counts.get(0, 0)/len(subset)*100:.2f}%)")
                  
        print("Fitting Word-level TF-IDF (n-grams: 1 to 2, max features: 25,000, stop words: english)...")
        word_vectorizer = TfidfVectorizer(
            ngram_range=(1, 2),
            max_features=25000,
            sublinear_tf=True,
            stop_words='english'
        )
        
        # Fit only on the training set to prevent data leakage!
        X_train_word = word_vectorizer.fit_transform(train_df['text'])
        X_test_word = word_vectorizer.transform(test_df['text'])
        
        print("Fitting Character-level TF-IDF (char n-grams: 3 to 5, max features: 25,000)...")
        char_vectorizer = TfidfVectorizer(
            analyzer='char',
            ngram_range=(3, 5),
            max_features=25000,
            sublinear_tf=True
        )
        
        X_train_char = char_vectorizer.fit_transform(train_df['text'])
        X_test_char = char_vectorizer.transform(test_df['text'])
        
        print(f"Word vocabulary size: {len(word_vectorizer.vocabulary_)}")
        print(f"Char vocabulary size: {len(char_vectorizer.vocabulary_)}")
        
        print("Concatenating Word and Character Feature Matrices...")
        X_train = hstack([X_train_word, X_train_char]).tocsr()
        X_test = hstack([X_test_word, X_test_char]).tocsr()
        
        y_train = train_df['label'].values
        y_test = test_df['label'].values
        
        print(f"Combined Training Matrix Shape: {X_train.shape}")
        print(f"Combined Testing Matrix Shape: {X_test.shape}")
        
        # Setup directories for this fold
        fold_output_dir = os.path.join(output_dir, f"fold_{fold}")
        fold_models_dir = os.path.join(models_dir, f"fold_{fold}")
        os.makedirs(fold_output_dir, exist_ok=True)
        os.makedirs(fold_models_dir, exist_ok=True)
        
        # Save sparse feature matrices
        save_npz(os.path.join(fold_output_dir, "X_train.npz"), X_train)
        save_npz(os.path.join(fold_output_dir, "X_test.npz"), X_test)
        
        # Save target label arrays
        np.save(os.path.join(fold_output_dir, "y_train.npy"), y_train)
        np.save(os.path.join(fold_output_dir, "y_test.npy"), y_test)
        
        # Save indices of train/test for traceability
        np.save(os.path.join(fold_output_dir, "train_idx.npy"), train_idx)
        np.save(os.path.join(fold_output_dir, "test_idx.npy"), test_idx)
        
        # Save the fitted vectorizers using joblib
        joblib.dump(word_vectorizer, os.path.join(fold_models_dir, "word_vectorizer.joblib"))
        joblib.dump(char_vectorizer, os.path.join(fold_models_dir, "char_vectorizer.joblib"))
        
        print(f"Fold {fold} preprocessing completed successfully!")


### 3. Execution Entry Point
Sets paths and runs the feature engineering pipeline.


In [ ]:
if __name__ == "__main__":
    INPUT_CSV = "train_v4_drcat_01_cleaned.csv"
    OUTPUT_DIR = "processed_data"
    MODELS_DIR = "models"
    
    run_preprocessing(INPUT_CSV, OUTPUT_DIR, MODELS_DIR)
